Carga y copia de dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/Usuario/Desktop/Proyecto_G07/dp261-g7/data/raw/10-employee_performance.csv")
df_clean = df.copy()

In [ ]:
Verifacion de nulos

In [2]:
print("Shape inicial:", df_clean.shape)
display(df_clean.head())
display(df_clean.dtypes)
display(df_clean.isnull().sum())
print("Duplicados:", df_clean.duplicated().sum())

Shape inicial: (14000, 20)


,employee_id,age,gender,education,department,job_level,years_at_company,years_in_current_role,monthly_salary,overtime_hours_monthly,num_projects_completed,performance_rating,training_hours_yearly,work_life_balance_score,distance_from_home_km,num_companies_worked,job_satisfaction,relationship_with_manager,stock_option_level,attrition
0,1,38,Female,Master,IT,3,6.3,0.2,6072.54,11,7,2,39.0,5,4.4,3,3,5,3,0
1,2,32,Male,Bachelor,Finance,1,3.4,3.8,3058.51,15,6,3,57.1,4,4.5,2,4,4,2,0
2,3,24,Female,Bachelor,Engineering,1,0.8,2.9,9270.48,5,6,5,41.1,1,49.6,2,3,1,0,1
3,4,33,Male,Bachelor,Sales,5,0.2,1.6,7567.36,3,8,3,21.2,2,0.7,2,5,4,1,0
4,5,51,Male,Bachelor,Finance,3,3.8,7.1,5621.87,5,8,4,42.6,4,6.6,3,3,4,2,1


employee_id                    int64
age                            int64
gender                           str
education                        str
department                       str
job_level                      int64
years_at_company             float64
years_in_current_role        float64
monthly_salary               float64
overtime_hours_monthly         int64
num_projects_completed         int64
performance_rating             int64
training_hours_yearly        float64
work_life_balance_score        int64
distance_from_home_km        float64
num_companies_worked           int64
job_satisfaction               int64
relationship_with_manager      int64
stock_option_level             int64
attrition                      int64
dtype: object

employee_id                  0
age                          0
gender                       0
education                    0
department                   0
job_level                    0
years_at_company             0
years_in_current_role        0
monthly_salary               0
overtime_hours_monthly       0
num_projects_completed       0
performance_rating           0
training_hours_yearly        0
work_life_balance_score      0
distance_from_home_km        0
num_companies_worked         0
job_satisfaction             0
relationship_with_manager    0
stock_option_level           0
attrition                    0
dtype: int64

Duplicados: 0


Mediana

In [3]:
##
num_cols = df_clean.select_dtypes(include="number").columns

for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

Moda

In [4]:
cat_cols = ["gender","education","department"]

for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

KNNImputer

In [6]:
from sklearn.impute import KNNImputer

knn_cols = ["age","monthly_salary","years_at_company"]
imputer = KNNImputer(n_neighbors=5)

df_clean[knn_cols] = imputer.fit_transform(df_clean[knn_cols])

In [7]:
print(df_clean[knn_cols].isnull().sum())

age                 0
monthly_salary      0
years_at_company    0
dtype: int64


IterativeImputer

In [8]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imp = IterativeImputer(random_state=42)

df_clean[knn_cols] = imp.fit_transform(df_clean[knn_cols])

In [9]:
print(df_clean[knn_cols].isnull().sum())

age                 0
monthly_salary      0
years_at_company    0
dtype: int64


Se aplicó imputación por mediana en variables numéricas por su robustez ante valores extremos y moda en categóricas. Se probaron KNN e IterativeImputer como métodos avanzados.

Tratamiento de Outliers

Variables sensibles: 
monthly_salary
distance_from_home_km
overtime_hours_monthly
training_hours_yearly

Método IQR

In [10]:
def winsor_iqr(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    
    low = q1 - 1.5*iqr
    high = q3 + 1.5*iqr
    
    df[col] = df[col].clip(lower=low, upper=high)

winsor_iqr(df_clean, "monthly_salary")
winsor_iqr(df_clean, "distance_from_home_km")

Z-score

In [11]:
from scipy.stats import zscore

z = np.abs(zscore(df_clean["monthly_salary"]))
outliers = (z > 3).sum()
print(outliers)

0


No eliminar filas,sino winsorizar

Duplicados de registros

In [12]:
print(df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates()

0


Duplicados de id de empleado

In [13]:
df_clean["employee_id"].duplicated().sum()

np.int64(0)

In [14]:
mask = df_clean["years_in_current_role"] > df_clean["years_at_company"]

df_clean.loc[mask, "years_in_current_role"] = df_clean.loc[mask, "years_at_company"]

In [15]:
(df_clean["age"] < 18).sum()

np.int64(0)

Correcion de tipo de datos

In [16]:
df_clean.dtypes

employee_id                    int64
age                          float64
gender                           str
education                        str
department                       str
job_level                      int64
years_at_company             float64
years_in_current_role        float64
monthly_salary               float64
overtime_hours_monthly         int64
num_projects_completed         int64
performance_rating             int64
training_hours_yearly        float64
work_life_balance_score        int64
distance_from_home_km        float64
num_companies_worked           int64
job_satisfaction               int64
relationship_with_manager      int64
stock_option_level             int64
attrition                      int64
dtype: object

In [17]:
cat_cols = ["gender","education","department"]

df_clean[cat_cols] = df_clean[cat_cols].astype("category")

In [18]:
df_clean["attrition"] = df_clean["attrition"].astype(int)

In [19]:
df_clean["employee_id"] = df_clean["employee_id"].astype(int)

Aplicacion de encoding a categorias

In [20]:
df_encoded = pd.get_dummies(
    df_clean,
    columns=["gender","department"],
    drop_first=True
)

In [21]:
orden = {
    "High School":1,
    "Bachelor":2,
    "Master":3,
    "PhD":4
}

df_encoded["education"] = df_clean["education"].map(orden)

In [22]:
target_map = df_clean.groupby("department")["attrition"].mean()

df_encoded["department_target"] = df_clean["department"].map(target_map)

Aplicacion de escalado

In [23]:
from sklearn.preprocessing import StandardScaler

cols = ["age","monthly_salary","distance_from_home_km"]

scaler = StandardScaler()
df_encoded[cols] = scaler.fit_transform(df_encoded[cols])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

mm = MinMaxScaler()
df_encoded[cols] = mm.fit_transform(df_encoded[cols])

In [24]:
from sklearn.preprocessing import RobustScaler

rb = RobustScaler()
df_encoded[cols] = rb.fit_transform(df_encoded[cols])

In [ ]:
Verificacion de antes/despues

In [25]:
print("Shape:", df_encoded.shape)
print("Nulls:", df_encoded.isnull().sum().sum())
print(df_encoded.dtypes)

Shape: (14000, 28)
Nulls: 0
employee_id                     int64
age                           float64
education                    category
job_level                       int64
years_at_company              float64
years_in_current_role         float64
monthly_salary                float64
overtime_hours_monthly          int64
num_projects_completed          int64
performance_rating              int64
training_hours_yearly         float64
work_life_balance_score         int64
distance_from_home_km         float64
num_companies_worked            int64
job_satisfaction                int64
relationship_with_manager       int64
stock_option_level              int64
attrition                       int64
gender_Male                      bool
gender_Non-binary                bool
department_Finance               bool
department_HR                    bool
department_IT                    bool
department_Legal                 bool
department_Marketing             bool
department_Operations 

In [26]:
from pathlib import Path

Path("C:/Users/Usuario/Desktop/Proyecto_G07/dp261-g7/data/interim").mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    "C:/Users/Usuario/Desktop/Proyecto_G07/dp261-g7/data/interim/employee_performance_clean.csv",
    index=False
)